# aw_02_a1 — Stage A1: PlayWorld SFT (Track A direct)

**Protocol**: §5.1 A1. Input: G3 frozen suites. Output: A1 adapter run + eval pair + analysis.

**Result summary (v0.4.x)**: A1 beats base on all 5 suites
(pass-rate Δ: adversarial +0.443, template/rule-OOD +0.193, ID +0.187, comp-OOD +0.120;
all paired-bootstrap CIs exclude 0, permutation p ≤ 0.0003).

**Failure-analysis narrative (archived below as x01–x08)**: the first A1 eval collapsed
(93% malformed_json) despite 95.5% train accuracy. An 8-step diagnostic chain
(template → adapter loading → truncation → completion learning → boundary tokens →
TRL rendering → trainer labels → dynamics/checkpoint) localized the root cause:
Qwen3's chat template injects an empty `<think>\n\n</think>\n\n` opener into every
assistant target, but Qwen3-8B-BASE has untrained think-token embeddings that
attention/MLP LoRA cannot repair. Fix: seed the opener at eval time
(`evaluation.opener_seed`, amendment v1.1) — no retrain needed.


In [ ]:
# @title common header
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

!git clone https://{os.environ["GITHUB_TOKEN"]}@github.com/m97j/axiom-world.git
%cd axiom-world
!pip install -e . -r requirements/colab-g4.lock.txt


In [ ]:
# @title a_a1_data — build SFT + prompt data (leakage-gated)
!python scripts/build_training_data.py


In [ ]:
# @title b_a1_train — A1 PlayWorld SFT
!python scripts/run_experiment.py \
  --config configs/experiments/a1_playworld_sft.yaml \
  --hf-sync-repo m97j/aw-runs-a1


In [ ]:
# @title c_a1_eval — A1 adapter, batched greedy (canonical profile)
!python scripts/build_eval_suites.py --episodes-per-suite 300

RUN_ID = "20260801-030335--a1-playworld-sft--s42--e24d72"  # <- update to your A1 run
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID}
print("\n".join(out))
adapter_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]

!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {adapter_dir} \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a1


In [ ]:
# @title d_a1_eval_control — raw base model
!python scripts/run_evaluation.py \
  --config configs/experiments/eval_playworld.yaml \
  --max-new-tokens 1024 --batch-size 100 \
  --hf-sync-repo m97j/aw-runs-a1


In [ ]:
# @title f_a1_analysis — paired comparison (bootstrap CI + permutation)
RUN_ID_a = "20260801-063425--eval-playworld--s42--3bf440"  # A1 eval run
RUN_ID_b = "20260801-071014--eval-playworld--s42--b7aed3"  # base eval run

!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID_a} --kind eval
!python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID_b} --kind eval

!python scripts/run_analysis.py \
  --run-a runs/{RUN_ID_a} --label-a a1-sft \
  --run-b runs/{RUN_ID_b} --label-b qwen3-8b-base \
  --output runs/{RUN_ID_a}/analysis_summary.json \
  --hf-sync-repo m97j/aw-runs-a1


---
## Failure Analysis (archived) — diagnostic chain x01–x08
Kept for provenance; not needed on the happy path. Each cell rejected one hypothesis:
x01 template mismatch → x02 adapter loading → x03 truncation → x04 completion learning
→ x05 boundary tokens (localized the pathology) → x06 TRL rendering → x07 trainer labels
→ x08 dynamics + checkpoint audit ⇒ root cause K (untrained think-opener under Base + LoRA).


In [ ]:
# @title x01_diagnose — chat-template consistency (hypothesis A)
RUN_ID = "20260801-030335--a1-playworld-sft--s42--e24d72"
out = !python scripts/fetch_run.py --repo m97j/aw-runs-a1 --run-id {RUN_ID}
adapter_dir = [line for line in out if line.startswith("ADAPTER_DIR=")][0].split("=", 1)[1]
!python scripts/inspect_sft_tokenization.py --run-id {RUN_ID} --num-samples 8


In [ ]:
# @title x02_adapter_effect — logit delta (hypothesis B)
!python scripts/diag_adapter_effect.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {adapter_dir}


In [ ]:
# @title x03_diag_token_lengths — truncation audit (hypothesis C)
!python scripts/diag_token_lengths.py \
  --config configs/experiments/eval_playworld.yaml \
  --prompt-file data/train/playworld_sft.jsonl \
  --caps 1024 4096


In [ ]:
# @title x04_diag_completion_learning — span-split accuracy (hypothesis D)
!python scripts/diag_completion_learning.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {adapter_dir} \
  --prompt-file data/train/playworld_sft.jsonl \
  --num-samples 8


In [ ]:
# @title x05_diag_first_token — boundary dissection (hypotheses E/F)
!python scripts/diag_first_token.py \
  --config configs/experiments/eval_playworld.yaml \
  --adapter-dir {adapter_dir} \
  --prompt-file data/train/playworld_sft.jsonl \
  --num-samples 4


In [ ]:
# @title x06_diag_trl_rendering — TRL vs eval rendering (hypothesis G, CPU-ok)
!python scripts/diag_trl_rendering.py \
  --config configs/experiments/eval_playworld.yaml \
  --prompt-file data/train/playworld_sft.jsonl \
  --num-samples 4


In [ ]:
# @title x07_diag_trainer_labels — processed tensors & labels (hypothesis H, CPU-ok)
!python scripts/diag_trainer_labels.py \
  --config configs/experiments/a1_playworld_sft.yaml \
  --prompt-file data/train/playworld_sft.jsonl \
  --num-samples 4


In [ ]:
# @title x08_diag_training_dynamics — overfit probe + checkpoint audit (hypotheses I/J)
!python scripts/diag_training_dynamics.py \
  --config configs/experiments/a1_playworld_sft.yaml \
  --prompt-file data/train/playworld_sft.jsonl \
  --adapter-dir {adapter_dir} \
  --steps 60
